# Part 1: Setting Up a SQLite Database

SQLite is a lightweight, file-based relational database that ships with Python's standard library — no server or installation required.

In this notebook we will:
- Connect to (and create) a SQLite database file
- Define a schema with three related tables
- Populate the tables with gene expression data
- Verify the structure using SQLite's built-in introspection

## Installation check

`sqlite3` is part of Python's standard library. No `pip install` needed.

In [1]:
import sqlite3
print(f"Python sqlite3 module version: {sqlite3.version}")
print(f"SQLite library version:        {sqlite3.sqlite_version}")

Python sqlite3 module version: 2.6.0
SQLite library version:        3.50.4


## Connect to a database

`sqlite3.connect()` opens an existing file or **creates a new one** if it does not exist.

In [2]:
DB_FILE = "gene_expression.db"

con = sqlite3.connect(DB_FILE)
con.execute("PRAGMA foreign_keys = ON")  # enforce foreign key constraints
print(f"Connected to: {DB_FILE}")

Connected to: gene_expression.db


## Create the schema

We model a simple gene expression experiment:

| Table | Description |
|---|---|
| `genes` | Gene metadata (symbol, chromosome, strand, genomic coordinates) |
| `samples` | Experimental samples (condition, tissue type) |
| `expression` | Expression values linking each gene to each sample |

`expression` is a **junction table** — it holds the many-to-many relationship between genes and samples.

In [3]:
# Drop existing tables so this notebook is re-runnable
con.executescript("""
    DROP TABLE IF EXISTS expression;
    DROP TABLE IF EXISTS samples;
    DROP TABLE IF EXISTS genes;
""")

# Create tables
con.executescript("""
CREATE TABLE genes (
    gene_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    symbol      TEXT    NOT NULL UNIQUE,
    chromosome  TEXT    NOT NULL,
    strand      TEXT    NOT NULL CHECK(strand IN ('+', '-')),
    start_pos   INTEGER NOT NULL,
    end_pos     INTEGER NOT NULL,
    description TEXT
);

CREATE TABLE samples (
    sample_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    name        TEXT    NOT NULL UNIQUE,
    condition   TEXT    NOT NULL,
    tissue      TEXT    NOT NULL
);

CREATE TABLE expression (
    expr_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    gene_id     INTEGER NOT NULL REFERENCES genes(gene_id),
    sample_id   INTEGER NOT NULL REFERENCES samples(sample_id),
    tpm         REAL    NOT NULL,
    raw_counts  INTEGER NOT NULL,
    UNIQUE(gene_id, sample_id)
);
""")

print("Tables created successfully.")

Tables created successfully.


## Inspect the schema

SQLite stores schema information in a special table called `sqlite_master`.

In [4]:
rows = con.execute("SELECT name, type FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
print("Tables in database:")
for name, kind in rows:
    print(f"  {kind}: {name}")

Tables in database:
  table: expression
  table: genes
  table: samples
  table: sqlite_sequence


In [5]:
# Show full CREATE statement for each table
for (name,) in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
    (sql,) = con.execute(f"SELECT sql FROM sqlite_master WHERE name=?", (name,)).fetchone()
    print(sql)
    print()

CREATE TABLE expression (
    expr_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    gene_id     INTEGER NOT NULL REFERENCES genes(gene_id),
    sample_id   INTEGER NOT NULL REFERENCES samples(sample_id),
    tpm         REAL    NOT NULL,
    raw_counts  INTEGER NOT NULL,
    UNIQUE(gene_id, sample_id)
)

CREATE TABLE genes (
    gene_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    symbol      TEXT    NOT NULL UNIQUE,
    chromosome  TEXT    NOT NULL,
    strand      TEXT    NOT NULL CHECK(strand IN ('+', '-')),
    start_pos   INTEGER NOT NULL,
    end_pos     INTEGER NOT NULL,
    description TEXT
)

CREATE TABLE samples (
    sample_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    name        TEXT    NOT NULL UNIQUE,
    condition   TEXT    NOT NULL,
    tissue      TEXT    NOT NULL
)

CREATE TABLE sqlite_sequence(name,seq)



## Insert seed data

In [6]:
GENES = [
    ("BRCA1",  "17", "+", 43044295,  43125483,  "Breast cancer type 1 susceptibility protein"),
    ("BRCA2",  "13", "+", 32315474,  32400266,  "Breast cancer type 2 susceptibility protein"),
    ("TP53",   "17", "-", 7661779,   7687550,   "Tumor protein p53"),
    ("EGFR",   "7",  "+", 55019021,  55207338,  "Epidermal growth factor receptor"),
    ("MYC",    "8",  "+", 127735434, 127742951, "MYC proto-oncogene"),
    ("PTEN",   "10", "-", 89622870,  89731687,  "Phosphatase and tensin homolog"),
    ("RB1",    "13", "-", 48303747,  48481890,  "Retinoblastoma protein"),
    ("KRAS",   "12", "-", 25205246,  25250929,  "KRAS proto-oncogene"),
    ("PIK3CA", "3",  "+", 179148114, 179240085, "Phosphatidylinositol 4,5-bisphosphate 3-kinase"),
    ("AKT1",   "14", "+", 104769349, 104795751, "AKT serine/threonine kinase 1"),
    ("HER2",   "17", "+", 39687914,  39730426,  "Human epidermal growth factor receptor 2"),
    ("CDH1",   "16", "-", 68737275,  68835822,  "E-cadherin"),
    ("VHL",    "3",  "+", 10141777,  10153669,  "Von Hippel-Lindau tumor suppressor"),
    ("MLH1",   "3",  "-", 36993459,  37050918,  "MutL homolog 1"),
    ("ATM",    "11", "+", 108222832, 108369102, "ATM serine/threonine kinase"),
]

con.executemany(
    "INSERT INTO genes (symbol, chromosome, strand, start_pos, end_pos, description) VALUES (?,?,?,?,?,?)",
    GENES
)
print(f"Inserted {len(GENES)} genes.")

Inserted 15 genes.


In [7]:
SAMPLES = [
    ("BRCA_T01", "tumor",  "breast"),
    ("BRCA_T02", "tumor",  "breast"),
    ("BRCA_T03", "tumor",  "breast"),
    ("BRCA_N01", "normal", "breast"),
    ("BRCA_N02", "normal", "breast"),
    ("LUNG_T01", "tumor",  "lung"),
    ("LUNG_T02", "tumor",  "lung"),
    ("LUNG_N01", "normal", "lung"),
    ("SKIN_T01", "tumor",  "skin"),
    ("SKIN_N01", "normal", "skin"),
]

con.executemany(
    "INSERT INTO samples (name, condition, tissue) VALUES (?,?,?)",
    SAMPLES
)
print(f"Inserted {len(SAMPLES)} samples.")

Inserted 10 samples.


In [ ]:
import random
random.seed(42)

gene_ids   = [r[0] for r in con.execute("SELECT gene_id FROM genes ORDER BY gene_id")]
sample_ids = [r[0] for r in con.execute("SELECT sample_id FROM samples ORDER BY sample_id")]

expr_rows = []
for gid in gene_ids:
    for sid in sample_ids:
        raw = random.randint(10, 2000)
        tpm = round(raw * random.uniform(0.5, 2.5), 2)
        expr_rows.append((gid, sid, tpm, raw))

con.executemany(
    "INSERT INTO expression (gene_id, sample_id, tpm, raw_counts) VALUES (?,?,?,?)",
    expr_rows
)
print(f"Inserted {len(expr_rows)} expression values ({len(gene_ids)} genes x {len(sample_ids)} samples).")

con.commit()

## Verify row counts

In [ ]:
for table in ("genes", "samples", "expression"):
    (count,) = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()
    print(f"  {table}: {count} rows")

## Preview each table

In [ ]:
import pandas as pd

pd.read_sql("SELECT * FROM genes", con)

In [ ]:
pd.read_sql("SELECT * FROM samples", con)

In [ ]:
pd.read_sql("SELECT * FROM expression LIMIT 10", con)

## Close the connection

Always close the connection when finished. The database is persisted to `gene_expression.db` on disk.

In [ ]:
con.close()
print("Connection closed. Database saved to gene_expression.db")

---
**Next:** Open `02_basic_queries.ipynb` to start writing SELECT queries.